## Overview 
Your team at EngageMetrics, a leading employee engagement analytics company, has just received datasets from multiple sources that need to be consolidated for an urgent executive presentation. 
The data includes employee feedback from regional offices in CSV format, educational achievement records in Excel spreadsheets, and compensation benchmarks from an external API. The leadership team needs insights on employee engagement trends by tomorrow morning.
<br>
The goal is to efficiently import and process this scattered data using Python. 
EngageMetrics datasets containing employee insights, educational records, and external market data, are used to work with the essential data import techniques needed to handle various file formats and common data challenges.

## Learning Outcomes 

- Import data from CSV files using pandas
- Load data from Excel files with proper formatting
- Connect to and retrieve data from a REST API
- Handle common data import challenges (missing values, inconsistent formats)

## Dataset Information 
Two different EngageMetrics data sources and an external API from Wikipedia:

1. <b>employee_insights.csv:</b> Employee performance and demographic data
2. <b>education_data.xlsx:</b> Educational background information
3. <b>Wikipedia REST API:</b> Income statistics in the United States

<b>1. imports</b>

In [38]:
import pandas as pd
import requests
from datetime import datetime
from pathlib import Path

<b>2. path adjustments</b>

In [39]:
BASE_DIR = Path.cwd().resolve().parent
DATA_DIR = BASE_DIR / "data"
employee_insights_path = DATA_DIR / "employee_insights.csv"
education_data_path = DATA_DIR / "education_data.xlsx"

<b>3. utils functions</b>

In [40]:
def convert_dates(df: pd.DataFrame, date_cols: list[str]) -> pd.DataFrame:
    valid_df = df.copy()
    for col in date_cols:
        valid_df[col] = pd.to_datetime(df[col], errors='coerce', format='mixed')
        invalid_mask = valid_df[col].isna() & df[col].notna()
        invalid_values: pd.DataFrame = df.loc[invalid_mask, col]
        if len(invalid_values) > 0:
            print(f'invalid values found at column {col}')
            print(invalid_values.sample(n=2))
            raise ValueError()

    return valid_df


def validate_year(value) -> bool:
    try:
        year = int(value)
        return 1900 <= year <= datetime.now().year
    except ValueError:
        return False

## Activities

### Activity 1: Regional Feedback Data Import

Import employee insights data from a CSV file. Import the quarterly employee engagement feedback data from all regions.

<b>Step 1:</b> Load data

In [41]:
employee_df = pd.read_csv(employee_insights_path)
employee_df.head(5)

,employee_id,age,salary,promotion_eligible,last_training_date,department,work_experience,projects_completed,hours_worked_weekly,work_mode,last_promotion_date,satisfaction_score,overtime_hours
0,E0001,54.0,NaN,NaN,15/08/2023,HR,NaN,14.0,NaN,remote work,2022-05-10,NaN,8.4
1,E0002,NaN,$64761,N,15/08/2023,NaN,1 years,NaN,53.3,HYBRID,05-10-2022,NaN,8.1
2,E0003,54.0,NaN,N,15/08/2023,Marketing,8,6.0,32.6,Hybrid,10/05/2022,10.0,5.2
3,E0004,NaN,NaN,No,NaN,NaN,16,1.0,37.8,Remote,05-10-2022,5.0,NaN
4,E0005,29.0,$61486,Y,15/08/2023,NaN,NaN,1.0,53.3,Hybrid,2022-05-10,NaN,0.3


<b>Step 2:</b> Check data quality

In [42]:
employee_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   employee_id          100 non-null    object 
 1   age                  44 non-null     float64
 2   salary               63 non-null     object 
 3   promotion_eligible   84 non-null     object 
 4   last_training_date   71 non-null     object 
 5   department           85 non-null     object 
 6   work_experience      71 non-null     object 
 7   projects_completed   48 non-null     float64
 8   hours_worked_weekly  67 non-null     float64
 9   work_mode            84 non-null     object 
 10  last_promotion_date  74 non-null     object 
 11  satisfaction_score   61 non-null     float64
 12  overtime_hours       70 non-null     float64
dtypes: float64(5), object(8)
memory usage: 10.3+ KB


In [43]:
employee_df.isna().sum()

employee_id             0
age                    56
salary                 37
promotion_eligible     16
last_training_date     29
department             15
work_experience        29
projects_completed     52
hours_worked_weekly    33
work_mode              16
last_promotion_date    26
satisfaction_score     39
overtime_hours         30
dtype: int64

In [44]:
# Identify inconsistent date formats 
date_cols = ['last_promotion_date', 'last_training_date']
employee_df = convert_dates(df=employee_df, date_cols=date_cols)

### Activity 2: Educational Records Import
Import educational background data from an Excel file. Process the talent development team's educational background data.

<b>Step 1:</b> Load data

In [45]:
education_df = pd.read_excel(education_data_path)
education_df.head(5)

,ID,graduation_year,educational_background
0,E0001,2011,Psychology
1,E0002,1995,Architecture
2,E0003,2007,Business Administration
3,E0004,2000,Business Administration
4,E0005,1991,Medicine


<b>Step 2:</b> Check data quality

In [46]:
education_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 3 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   ID                      100 non-null    object
 1   graduation_year         100 non-null    int64 
 2   educational_background  100 non-null    object
dtypes: int64(1), object(2)
memory usage: 2.5+ KB


In [47]:
education_df.isna().sum()

ID                        0
graduation_year           0
educational_background    0
dtype: int64

In [48]:
# Verify whether graduation years are in the correct format
invalid_years = education_df[~education_df['graduation_year'].apply(validate_year)]
print('invalid years found:')
print(invalid_years)

invalid years found:
Empty DataFrame
Columns: [ID, graduation_year, educational_background]
Index: []


In [49]:
# Check if there are any missing educational backgrounds
education_df[education_df["educational_background"].isna()]

,ID,graduation_year,educational_background


### Activity 3: Compensation Benchmark Data Import
Retrieve data from the Wikipedia API. Retrieve industry compensation data for comparison.

<b>Step 1:</b> Make an API request

In [50]:
url = "https://en.wikipedia.org/api/rest_v1/page/summary/Income_in_the_United_States"

try:
    response = requests.get(url)
except requests.RequestException as e:
    print(f'error:{e}')
    exit(1)

# Parse the JSON response
if response.status_code == 200:
    data = response.json()
else:
    print(f'response error status: {response.status_code}')
    print(response)
# YOUR CODE HERE 

response error status: 403
<Response [403]>
